# RanPAC Critique A · Stress-test *First-Task Bias* của Phase 1 (PETL)

Notebook này thuộc phần **Critical Analysis & Empirical Demonstration** trên **Split CIFAR-100** (`seed = 1993`), kiểm chứng trực tiếp điểm yếu thiết kế cốt lõi nhất của **Algorithm 1 (RanPAC)**:

> **Giả định ẩn của Phase 1:** RanPAC chỉ huấn luyện nhánh **AdaptFormer** trong 20 epoch trên **duy nhất Task 1**, sau đó đóng băng vĩnh viễn cho toàn bộ các task về sau (`Task 2..T`). Bài báo mặc định chọn `Task 1` gồm `B = 10` (hoặc `20`) lớp lấy ngẫu nhiên đồng đều trên toàn tập dữ liệu, vô tình đảm bảo Task 1 có độ đa dạng ngữ nghĩa cao.

### Thiết kế 4 kịch bản đối chứng (cùng 100 lớp CIFAR-100, có cặp đối chứng `No Phase 1` tương ứng)

| Mã kịch bản | Cấu trúc Task (`sum = 100` lớp) | Bản chất kiểm chứng |
| --- | --- | --- |
| `b10-random` | `10` lớp đầu ngẫu nhiên + `9 × 10` lớp sau | **Chuẩn bài báo:** Task 1 đủ lớn (5.000 ảnh) và đa dạng ngữ nghĩa. |
| `b5-narrow` | `5` lớp đầu + `19 × 5` lớp sau (`20` task) | **Task 1 hẹp vừa:** giảm một nửa số lớp và dữ liệu thích nghi ban đầu (2.500 ảnh). |
| `b2-narrow` | `2` lớp đầu + `7 × 14` lớp sau (`8` task) | **Task 1 rất hẹp:** chỉ 2 lớp ở phiên đầu (1.000 ảnh) — kiểm tra hiện tượng overfit cục bộ của adapter. |
| `b10-flora` | `10` lớp thực vật (*Hoa + Cây gỗ*) + `9 × 10` lớp khác miền | **Semantic-shift Task 1:** đủ 10 lớp (5.000 ảnh) nhưng hẹp về miền ngữ nghĩa so với 90 lớp động vật, phương tiện, đồ vật ở các task sau. |

Với **mỗi kịch bản**, pipeline chạy song song cả bản **`No Phase 1` (Frozen ViT-B/16)** trên chính cùng `class_order` và `task_sizes` (tái phân hoạch tức thời từ cache sẵn có, tốn 0 giây trích đặc trưng GPU) để tách bạch chính xác:
- **Mức tăng trên Task 1 ($\Delta_{\text{Task 1}}$):** độ dẻo (*plasticity*) tại phiên đầu.
- **Mức chuyển giao sang các lớp chưa thấy ($\Delta_{\text{Later}}$ trên `100 - B` lớp sau):** kiểm tra xem lợi ích của Phase 1 có sụp đổ hoặc gây *chuyển giao âm (negative transfer)* khi Task 1 hẹp/lệch miền hay không.


## 1. Cài đặt thư viện và kết nối Google Drive

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py`, `ranpac_critique.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Kết quả được lưu checkpoint từng epoch/kịch bản và log lên W&B `pnp123/RanPAC-reproduction` với tag `['empirics', 'critique', 'first-task-bias']`.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import gc
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

required = (
    'ranpac_core.py', 'ranpac_data.py', 'ranpac_colab.py',
    'ranpac_full.py', 'ranpac_critique.py', 'test_ranpac_core.py',
)
CODE_DIR = SOURCE_DIR / 'src' if (SOURCE_DIR / 'src' / 'ranpac_core.py').is_file() else SOURCE_DIR
missing = [name for name in required if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu {", ".join(missing)} trong {CODE_DIR}')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from ranpac_core import SEED
from ranpac_critique import (
    CIFAR100_FINE_CLASSES, build_first_task_regimes, run_first_task_regime,
)
from ranpac_data import atomic_output


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình thí nghiệm và kiểm tra danh sách lớp Task 1

- Mặc định Phase 2 dùng `M = 10000` và `FIXED_LAMBDA = 100.0` (như Table A5, chênh lệch `< 0.1 pp` so với CV nhưng nhanh gấp hàng chục lần; đặt `FIXED_LAMBDA = None` nếu muốn bật 80/20 Task-CV).
- Nếu đã có checkpoint `full-s1993-main/checkpoint.pt` từ [04_ranpac_full.ipynb](04_ranpac_full.ipynb), `REUSE_NB04_ADAPTER = True` sẽ tái sử dụng ngay trọng số AdaptFormer 20 epoch cho mốc chuẩn `b10-random` (không cần train lại).


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'          # 'online', 'offline', hoặc 'disabled'
RUN_LABEL = 'ftb-01'

SELECTED_REGIMES = ('b10-random', 'b5-narrow', 'b2-narrow', 'b10-flora')
PHASE1_EPOCHS = 20
MICROBATCH = 48
PRECISION = 'fp32'

M = 10_000
FIXED_LAMBDA = 100.0           # Đặt None nếu muốn chọn lambda bằng 80/20 CV theo từng task
FEATURE_BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

REUSE_NB04_ADAPTER = True
NB04_CKPT = WORK_DIR / 'runs' / f'full-s{SEED}-main' / 'checkpoint.pt'

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trên Colab trước khi chạy Phase 1.')

regimes = build_first_task_regimes(SEED)
preview_rows = []
for key in SELECTED_REGIMES:
    spec = regimes[key]
    k1 = spec['task_sizes'][0]
    t1_names = [CIFAR100_FINE_CLASSES[idx] for idx in spec['class_order'][:k1]]
    preview_rows.append({
        'regime': key,
        'first_task_B': k1,
        'num_tasks': len(spec['task_sizes']),
        'task1_classes': ', '.join(t1_names),
    })
display(pd.DataFrame(preview_rows))


## 3. Thực thi 4 kịch bản First-Task Bias

Mỗi kịch bản tự lưu checkpoint sau từng epoch Phase 1; nếu phiên Colab bị ngắt giữa chừng, chạy lại cell dưới với cùng `RUN_LABEL` sẽ tự động nối tiếp từ epoch dở dang.


In [ ]:
regime_outputs = {}
for key in SELECTED_REGIMES:
    ckpt = run_first_task_regime(
        key, WORK_DIR,
        regimes=regimes,
        run_label=RUN_LABEL,
        phase1_epochs=PHASE1_EPOCHS,
        microbatch=MICROBATCH,
        precision=PRECISION,
        projection_dim=M,
        fixed_lambda=FIXED_LAMBDA,
        feature_batch_size=FEATURE_BATCH_SIZE,
        stats_device=STATS_DEVICE,
        workers=NUM_WORKERS,
        data_root=DATA_ROOT,
        reuse_nb04_adapter_path=(NB04_CKPT if REUSE_NB04_ADAPTER else None),
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        wandb_mode=WANDB_MODE,
    )
    regime_outputs[key] = ckpt
    gc.collect()
    torch.cuda.empty_cache()


## 4. Bảng tổng hợp và Phân rã Độ dẻo Task 1 vs Khả năng tổng quát hóa sang Task sau

- `overall_100_pct`: độ chính xác trên toàn bộ 10.000 ảnh test của 100 lớp CIFAR-100 sau khi học hết mọi task (đo công bằng tuyệt đối bất kể số lượng task là 8, 10 hay 20).
- `gain_task1_pp` ($\Delta_{\text{Task 1}}$): chênh lệch độ chính xác trên các lớp thuộc Task 1 giữa `Có Phase 1` và `Không Phase 1`.
- `gain_later_pp` ($\Delta_{\text{Later}}$): chênh lệch độ chính xác trên `100 - B` lớp xuất hiện ở các task sau (`Task 2..T`) — chỉ số phản ánh trực tiếp **First-Task Bias**.


In [ ]:
df_ftb = pd.DataFrame([regime_outputs[k]['summary'] for k in SELECTED_REGIMES])
summary_dir = WORK_DIR / 'sweeps' / f'first-task-bias-s{SEED}-{RUN_LABEL}'
with atomic_output(summary_dir / 'first_task_bias_summary.csv') as tmp:
    df_ftb.to_csv(tmp, index=False)

cols_show = [
    'regime', 'first_task_classes', 'frozen_overall_100_pct', 'phase1_overall_100_pct',
    'gain_overall_pp', 'phase1_task1_pct', 'phase1_later_pct',
    'gain_task1_pp', 'gain_later_pp',
]
display(df_ftb[cols_show].round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout='constrained')
x = np.arange(len(df_ftb))
width = 0.36

# Panel 1: Overall 100-class accuracy (No Phase 1 vs With Phase 1)
ax = axes[0]
b1 = ax.bar(x - width / 2, df_ftb['frozen_overall_100_pct'], width,
            label='No Phase 1 (Frozen ViT + RP)', color='#94a3b8')
b2 = ax.bar(x + width / 2, df_ftb['phase1_overall_100_pct'], width,
            label='With Phase 1 (AdaptFormer Task 1 + RP)', color='#2563eb')
ax.set_xticks(x)
ax.set_xticklabels(df_ftb['regime'], rotation=12)
ax.set_ylabel('Overall 100-class Accuracy (%)')
ax.set_title('Hiệu năng tổng thể 100 lớp: Có vs Không có Phase 1')
ax.set_ylim(max(75.0, df_ftb['frozen_overall_100_pct'].min() - 4.0),
            min(100.0, df_ftb['phase1_overall_100_pct'].max() + 3.0))
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='lower left', fontsize=9)
for bar in (*b1, *b2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8.5)

# Panel 2: Plasticity vs Generalization Gain (Delta Task 1 vs Delta Later Tasks)
ax2 = axes[1]
g1 = ax2.bar(x - width / 2, df_ftb['gain_task1_pp'], width,
             label='Δ trên lớp Task 1 (Plasticity)', color='#f59e0b')
g2 = ax2.bar(x + width / 2, df_ftb['gain_later_pp'], width,
             label='Δ trên các lớp Task 2..T (Transfer)', color='#10b981')
ax2.axhline(0.0, color='#1e293b', linewidth=1.0, linestyle='--')
ax2.set_xticks(x)
ax2.set_xticklabels(df_ftb['regime'], rotation=12)
ax2.set_ylabel('Mức tăng so với No Phase 1 (pp)')
ax2.set_title('First-Task Bias: Tăng mạnh ở Task 1 nhưng suy hao ở Task sau')
ax2.grid(axis='y', alpha=0.3)
ax2.legend(loc='best', fontsize=9)
for bar in (*g1, *g2):
    h = bar.get_height()
    va = 'bottom' if h >= 0 else 'top'
    offset = 0.15 if h >= 0 else -0.15
    ax2.text(bar.get_x() + bar.get_width() / 2, h + offset,
             f'{h:+.2f}', ha='center', va=va, fontsize=8.5)

fig_path = summary_dir / 'first_task_bias_analysis.png'
with atomic_output(fig_path) as tmp:
    fig.savefig(tmp, dpi=180, format='png')
plt.show()
print(f'Đã lưu bảng và biểu đồ tại: {summary_dir}')
